# Ch 3: Building Attention Mechanism
- In the previous chapter, we studied about preprocessing the text to embeddings to make it suitable for training. But the question arises, how we can find the relationship between these embeddings, or words in a sentence.
- **Self Attention** refers to,  how much importance or attention does a word in a sentence give to all words(including itself) in a sentence. 

- Example: `"The cat ate fish"`

|Word(Query)|Attends to|Conclusion|
|-|-|-|
|"The"|"cat"|Emphasizes on cat|
|"cat"|"ate"|Action done by cat|
|"ate"|"cat","fish"|Pays attention to the actor("cat") and object("fish") on which action is performed|
|"fish"|"ate"|The action done on object|

- Finding the attention weights among different words in a sentence, helps the machine to understand the meaning of the sentence.
- GPTs uses Multihead Causal Attention mechanism to capture the meaning of the sentence, but we have to first understand, how do we evolved to achieve this attention mechanism.
- We will learn attention mechanism in this flow:

<p align="center"><img src="Images/Screenshot 2025-06-17 100600.png" width="" height=""></p>

- We will experiment with the sentence "Your journey starts with one step" and find attention weights among them.

- Let us find the embeddings for this sentence first

In [1]:
import torch
import tiktoken
torch.manual_seed(123)# To prevent changing values on rerunning
# Tokenizing text
tokenizer=tiktoken.get_encoding('gpt2')
text="Your journey starts with one step"
print(f'Given sentence:\n"{text}"')
token_ids=tokenizer.encode(text,allowed_special={"<|endoftext|>"})
# Getting token Embeddings
vocab_size=50257
output_dim=3
token_embedding_layer=torch.nn.Embedding(vocab_size,output_dim)
token_embeddings=token_embedding_layer(torch.tensor(token_ids))
# Getting positional embeddings
window_size=len(token_ids)
pos_embedding_layer=torch.nn.Embedding(window_size,output_dim)
pos_embeddings=pos_embedding_layer(torch.arange(window_size))
# Getting total embeddings
input_embeddings=token_embeddings+pos_embeddings
print("Total embeddings:\n",input_embeddings)

Given sentence:
"Your journey starts with one step"
Total embeddings:
 tensor([[ 0.8311,  1.3393, -0.8587],
        [ 2.1752, -0.5277, -2.3310],
        [ 2.2730,  1.0514, -0.6150],
        [ 1.2780, -2.2226, -1.3328],
        [-2.6924,  3.0901,  0.6925],
        [-0.7583,  0.3646, -0.9988]], grad_fn=<AddBackward0>)


- So we got the embeddings for our sentence "Your journey starts with one step" as given above.

**Convention for Input Data Representation**
- By convention the input data for LLM is a 3-Dimentional data and these 3 dimentions are:
    1. **Dim 0:** Number of batches 
    2. **Dim 1:** Number of tokens
    3. **Dim 2:** Dimention size of embedding
- In the `input_batch` given below , it's shape is `[2,6,3]`,which means it has 2 batches,6 tokens and 3-dimentional embedding vector. 

In [2]:
input_batch=torch.stack([input_embeddings,input_embeddings])
print(input_batch.shape)

torch.Size([2, 6, 3])


**Basic Terminologies**
1. **Query:** It is the token, whose attentions scores are computed with respect to all the tokens in a sentence(including itself).
2. **Attention scores:** It is the matrix formed as a result of quering each word to each other, which describes about the correlation among words.
3. **Attention weights:** If the attention score matrix is normalized in rows, we get attention weights, which gives a better interpretation about the correlation.
4. **Context Vector:** It is the matrix representing meaning of each word by multiplying attention weights with values of each word.

**Significance of Attention Weights**
- It determines how much a particular ward in a sentence focusses on different words in the sentence.
- Eg, In the sentence `The lion chased the deer.`,  the model focuses most on `lion` and `deer` while understanding `chased`, because they are the subject and object of the action.

|**Token**|	**Attention Weight for query="chase"**|
|-|-|
|The|0.05|
|lion|0.40|
|chased|0.10|
|the|0.05|
|deer|0.40|

**Difference between Context Vector and Embedding**
- Embeddings tells about the general or dictionary meaning of the word, whereas context vector tells about the meaning of the word with respect to the sentence

- Example: 
    - In the given sentence find the meaning of the word `bank`
    - The embedding for `bank` may reffer to the dictionary meaning, which captures meaning of the word

        Dictionary Meaning of `bank` -> ["river edge","financial institution"]

- The context vector changes depending on the sentence:
    |Sentence|Context Vector meaning for "bank"|
    |-|-|
    |`"He sat by the river bank"`|river edge|
    |`"She went to the bank to deposit money"`|financial institution|
   


In [3]:
input_batch=torch.stack((input_embeddings,input_embeddings),dim=0)
print(input_batch.shape)

torch.Size([2, 6, 3])


## Simplified Self Attention
- This model is a theoritical model, not for any practical use.
- This model basically demonstrates how self attention work, without involving trainable attention weights.

**Algorithm**
1. Find `attention_scores` by multiplying `input_embeddings` with its transpose
2. Finding `attention_weights` by apply softmax accross rows in `attention_scores`
3. Finding `context_vectors` by multiplying `attention_weights` with `input_embeddings`

**Drawbacks**
- Theoritical model, just for demonstrating self attention
- No trainable weight

In [4]:
# Finding Attention scores
attn_scores_1=input_batch@input_batch.transpose(1,2)
print("Attention Scores Shape: \n",attn_scores_1.shape)
# Finding Attention weights
attn_weights_1=torch.softmax(attn_scores_1,dim=-1)
print("Attention Weights: \n",attn_weights_1.shape)
# Finding Context vectors
context_vectors_1=attn_weights_1@input_batch
print("Context vectors: \n",context_vectors_1.shape)

Attention Scores Shape: 
 torch.Size([2, 6, 6])
Attention Weights: 
 torch.Size([2, 6, 6])
Context vectors: 
 torch.Size([2, 6, 3])


## Self Attention Mechanism
- In the complete implementation of Self Attention we include trainable weights like: $W_q$,$W_k$ and $W_v$
    - $W_q$: For handling query vector
    - $W_k$: For handling key vector
    - $W_v$: For handling value vector
- Here we use 3 vectors:
    1. **Query vector($Q$)**  represents what each token wants to find in other tokens.

    2. **Key vector($K$)** tells about what does each token has to offer which the query vector is looking for.
    3. **Value vector($V$)** used deriving the context to be displayed.
- For example, when you search for videos on Youtube, the search engine will map your query (text in the search bar) against a set of keys (video title, description, etc.) associated with candidate videos in their database, then present you the best matched videos (values)
- **NOTE:** In GPTs dimention of context vector and embeddings are generally same.
- For input embedding $X$ and $d_k$ being the desired dimention of key vector
    - Query vector($Q$)=$XW_q$
    - Key vector($K$)=$XW_k$
    - Value vector($V$)=$XW_v$
    - Attention scores= $QK^T$
    - Attention weights($W$)= Softmax(Attention score/$\sqrt{d_k}$,dim=-1)
    - Context Vectors= $WV$
- **NOTE:** We divide the Attention score by $\sqrt{d_k}$, because as $d_k$ increases, dot products become very large in magnitude and when passed to softmax we get sharp distribution. To prevent this we divide by $\sqrt{d_k}$

**Algorithm**
1. Initialize `W_q`,`W_k` and  `W_v` in `__init__` function.
2. Find the query,key and value vectors
3. Find attention scores 
4. Find attention weights
5. Find context vectors


In [3]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, output_dim):
        super().__init__()
        # Initialize W_q,W_k and W_v
        self.W_query = nn.Parameter(torch.rand(output_dim, output_dim))
        self.W_key   = nn.Parameter(torch.rand(output_dim, output_dim))
        self.W_value = nn.Parameter(torch.rand(output_dim, output_dim))

    def forward(self, x):
        # Find the query,key and value vectors
        queries = torch.matmul(x, self.W_query) 
        keys    = torch.matmul(x, self.W_key)    
        values  = torch.matmul(x, self.W_value) 
        # Find attention scores
        attn_scores = queries @ keys.transpose(1,2) # omega
        # Find attention weights
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        # Find context vectors
        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(output_dim)
print(sa_v1(input_batch))

tensor([[[ 0.9738,  1.2085,  2.1569],
         [ 0.2429,  0.0610,  0.7072],
         [ 1.1155,  1.4300,  2.4240],
         [-0.0372, -0.4598, -0.0819],
         [ 1.1081,  1.4195,  2.3994],
         [ 0.1722, -0.0619,  0.4991]],

        [[ 0.9738,  1.2085,  2.1569],
         [ 0.2429,  0.0610,  0.7072],
         [ 1.1155,  1.4300,  2.4240],
         [-0.0372, -0.4598, -0.0819],
         [ 1.1081,  1.4195,  2.3994],
         [ 0.1722, -0.0619,  0.4991]]], grad_fn=<UnsafeViewBackward0>)


- We replace the `Parameter` layer with `Linear` for the following reasons:
    - To include bias
    - Automatic weight initialisation
    - Reducing computations

- By convention, in attention mechanisms different batches used same trainable paramenters for:
    1. Better Generalisation
    2. Better performance with less use of trainable parameters



In [6]:
class SelfAttention_v2(nn.Module):
    def __init__(self,output_dim, qkv_bias=False):
        super().__init__()
        # Initialize W_q,W_k and W_v
        self.W_query = nn.Linear(output_dim, output_dim, bias=qkv_bias)
        self.W_key   = nn.Linear(output_dim, output_dim, bias=qkv_bias)
        self.W_value = nn.Linear(output_dim, output_dim, bias=qkv_bias)

    def forward(self, x):
        # Find the query,key and value vectors
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        # Find attention scores
        attn_scores = queries @ keys.transpose(1,2)
        # Find attention weights
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        # Find context vectors
        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v2 = SelfAttention_v2(output_dim)
print(sa_v2(input_batch))

tensor([[[-0.3511, -0.2821,  0.1325],
         [-0.3961, -0.3828,  0.1463],
         [-0.2188, -0.2444,  0.2523],
         [-0.4629, -0.5163,  0.1399],
         [-0.4258, -0.2331,  0.0037],
         [-0.4822, -0.3966,  0.0578]],

        [[-0.3511, -0.2821,  0.1325],
         [-0.3961, -0.3828,  0.1463],
         [-0.2188, -0.2444,  0.2523],
         [-0.4629, -0.5163,  0.1399],
         [-0.4258, -0.2331,  0.0037],
         [-0.4822, -0.3966,  0.0578]]], grad_fn=<UnsafeViewBackward0>)


## Causal Attention
- GPTs are autoregressive model , which means it predicts the next token in a sequence by using only the previously generated tokens as input.
- As the GPT queries each token from left to right of a sequence, we should ensure that while querying a particular token it should only be aware about the previously explored token.
- If the model used non-causal self-attention, it would require the entire output to be known before generating anything — which breaks the whole idea of generation. It losses it's text generation ability just like BERT.
- Just like if you cheat in exam, you have depend on cheats everytime when exam comes,  but you actually don't known anything about the subject.
- Other than masking of future tokens , we also add dropout layer to prevent overfitting.
- Now let's understand about how we find Causal attention.

In [7]:
queries=sa_v2.W_query(input_batch[0])
keys=sa_v2.W_key(input_batch[0])
attn_scores=queries@keys.T
print("Unmasked Attention scores:\n",attn_scores)

Unmasked Attention scores:
 tensor([[ 1.2208,  0.9214,  1.7753, -0.5408,  0.3478, -0.1268],
        [-0.0426, -0.0370,  0.2057,  0.0253, -0.3430, -0.2686],
        [ 1.5050,  1.1347,  1.4574, -0.6992,  1.3476,  0.5859],
        [-1.5492, -1.1744, -2.0074,  0.6915, -0.7443, -0.0898],
        [ 1.6794,  1.2741,  3.0843, -0.7099, -0.3338, -0.8246],
        [-0.1069, -0.0808,  0.6577,  0.0820, -1.0508, -0.8146]],
       grad_fn=<MmBackward0>)


- Now let us mask the future tokens with `-torch.inf` in attention scores.

In [8]:
context_length=input_batch.shape[1]
mask=torch.triu(torch.ones(context_length,context_length),diagonal=1)
print('Mask:\n',mask)


Mask:
 tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])


In [9]:
attn_scores_masked=attn_scores.masked_fill(mask.bool(),-torch.inf)
print("Masked attention scores:\n",attn_scores_masked)

Masked attention scores:
 tensor([[ 1.2208,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.0426, -0.0370,    -inf,    -inf,    -inf,    -inf],
        [ 1.5050,  1.1347,  1.4574,    -inf,    -inf,    -inf],
        [-1.5492, -1.1744, -2.0074,  0.6915,    -inf,    -inf],
        [ 1.6794,  1.2741,  3.0843, -0.7099, -0.3338,    -inf],
        [-0.1069, -0.0808,  0.6577,  0.0820, -1.0508, -0.8146]],
       grad_fn=<MaskedFillBackward0>)


- Now, we will apply softmax on `attn_scores_masked` to get `attn_weights`

In [10]:
attn_weights=torch.softmax(attn_scores_masked/keys.shape[-1]**0.5,dim=-1)
print('Attention weights:\n',attn_weights)

Attention weights:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4992, 0.5008, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3597, 0.2904, 0.3499, 0.0000, 0.0000, 0.0000],
        [0.1503, 0.1866, 0.1153, 0.5479, 0.0000, 0.0000],
        [0.2171, 0.1718, 0.4886, 0.0546, 0.0679, 0.0000],
        [0.1686, 0.1712, 0.2622, 0.1881, 0.0978, 0.1121]],
       grad_fn=<SoftmaxBackward0>)


- Now we will apply dropout to our `attention_weights` to prevent. 
- We can see below which part of the matrix did the dropout layer masked, when we apply dropout layer on `demo`

In [11]:
torch.manual_seed(123)
dropout=nn.Dropout(0.5)
demo=torch.ones(context_length,context_length)
print("Masked area of dropout layer:\n",dropout(demo))
attn_weights_dropped=dropout(attn_weights)
print("Attention weights after applying dropout:\n",attn_weights_dropped)

Masked area of dropout layer:
 tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])
Attention weights after applying dropout:
 tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5809, 0.6998, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.2307, 1.0957, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.1358, 0.0000],
        [0.3373, 0.3424, 0.0000, 0.3761, 0.1956, 0.2241]],
       grad_fn=<MulBackward0>)


- The number which are not masked are multiplied by $\frac{1}{\text{dropout rate}}$ to compensate, the loss of total value.
- Now we will find context vectors as usual.

In [12]:
values=sa_v2.W_value(input_batch[0])
context_vectors_demo=attn_weights_dropped@values
print("Context vectors:\n",context_vectors_demo)

Context vectors:
 tensor([[-0.4311, -0.3554,  0.4982],
        [ 0.0000,  0.0000,  0.0000],
        [-1.0376, -0.6196, -0.2063],
        [-1.3226, -1.0321, -0.4906],
        [ 0.1659,  0.0793,  0.1478],
        [-0.7726, -0.8287,  0.2423]], grad_fn=<MmBackward0>)


**Algorithm**
1. Define `W_q`,`W_k`,`W_v`,`dropout` and `mask` in `__init__`(`mask` must be declared inside `register_buffer()` to prevent device mismatch)
2. Find queries,keys and values
3. Find attention scores and apply mask
4. Find attention weights
5. Find context vectors

In [13]:
class CausalAttention(nn.Module):
    def __init__(self,dim,dropout,context_length,qkv_bias=False):
        super().__init__()
        # Define W_q,W_k,W_v,dropout and mask 
        self.W_q=nn.Linear(dim,dim,bias=qkv_bias)
        self.W_k=nn.Linear(dim,dim,bias=qkv_bias)
        self.W_v=nn.Linear(dim,dim,bias=qkv_bias)
        self.dropout=nn.Dropout(dropout)
        # To prevent device mismatch
        self.register_buffer('mask',torch.triu(torch.ones(context_length,context_length),diagonal=1))
        
    def forward(self,x):
        # Find queries,keys and values
        queries=self.W_q(x)
        keys=self.W_k(x)
        values=self.W_v(x)
        # Find attention scores and apply mask
        attn_scores=queries@keys.transpose(1,2)
        attn_scores=attn_scores.masked_fill(self.mask.bool(),-torch.inf)
        # Find attention weights and apply dropout
        attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
        attn_weights=self.dropout(attn_weights)
        # Find context vectors
        context_vectors=attn_weights@values
        return context_vectors

torch.set_printoptions(sci_mode=False)
torch.manual_seed(123)
ca=CausalAttention(3,0.1,input_batch.shape[1])
context_vectors=ca(input_batch)
print(context_vectors)    


tensor([[[    -0.2395,     -0.1975,      0.2768],
         [    -0.8683,     -0.7376,      0.1025],
         [    -0.6626,     -0.4153,     -0.0151],
         [    -1.0497,     -0.8411,     -0.2443],
         [    -0.4838,     -0.2393,     -0.0253],
         [    -0.5358,     -0.4406,      0.0642]],

        [[    -0.2395,     -0.1975,      0.2768],
         [    -0.7488,     -0.6390,     -0.0357],
         [    -0.6626,     -0.4153,     -0.0151],
         [    -1.0497,     -0.8411,     -0.2443],
         [    -0.2269,     -0.0201,     -0.0131],
         [    -0.5063,     -0.3683,     -0.0006]]],
       grad_fn=<UnsafeViewBackward0>)


- It is important to use in `register_buffer()` for non trainable tensors declared in `__init__()` the following reasons:
    - All non trainable tensors declared in `__init__()` are assigned to `cpu` by default
    - If we change the device of a `model` object inherited from `nn.Module` class using `model.to(device)`, all trainable parameters like(`nn.Linear()`,`nn.Parameters()`,etc) are assigned to the `device` by default, but non-trainable tensors like(`torch.tensor()`,`torch.ones()`,`torch.triu()`,etc) still assigned to `cpu`
    - To assign these non-trainable tensors to the desired `device`, we should save it `register_buffer()`
- In the the code sample given below, shows that in `ModelWithoutBuffer` we got a device mismatch error, but in `ModelWithBuffer` we got no such error. Hence, it is a wise choice to pass non trainable tensors to `register_buffer()`, just like we did in `CausalAttention`.
- You can see below , how trainable parameters like `nn.Linear()` are automatically assigned to the device 

In [14]:
class ModelWithoutBuffer(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.var=torch.rand(dim,dim) # Non Trainable parameter
        self.w1=nn.Linear(dim,dim) # Trainable parameter
    def forward(self,input):
        print("Non trainable parameter device: ",self.var.device)
        print("Trainable parameter device: ",self.w1.weight.device)
        try: # Trying to use non trainable parameter on input
            x=input*self.var
            print("Multiplication successfull!!")
        except Exception as e:
            print(e)

x=torch.ones(2,4,4,device='cuda')
model1=ModelWithoutBuffer(x.shape[1])
model1.to('cuda')
model1(x)

Non trainable parameter device:  cpu
Trainable parameter device:  cuda:0
Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!


In [15]:
class ModelWithBuffer(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.register_buffer('var',torch.rand(dim,dim)) # Don't belong to nn class
        self.w1=nn.Linear(dim,dim) # Belong to nn class
    def forward(self,input):
        print("Non trainable parameter device: ",self.var.device)
        print("Trainable parameter device: ",self.w1.weight.device)
        try: # Trying to use non trainable parameter on input
            x=input*self.var
            print("Multiplication successfull!!")
        except Exception as e:
            print(e)
model2=ModelWithBuffer(x.shape[1])
model2.to('cuda')
model2(x)

Non trainable parameter device:  cuda:0
Trainable parameter device:  cuda:0
Multiplication successfull!!


- Even if we set `requires_grad=True` for a tensor, it is still not trainable because it is not registered in `model.parameters()`.
- Hence , in the code given below we could that `self.wrong_way` did not get assigned to device automatically and not listed in `model.parameters()` as.
- To make a tensor trainable, we need to pass it in `nn.Parameter` as we did in `self.correct_way`

In [16]:

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.wrong_way = torch.randn(1, 2, requires_grad=True)  # Not registered
        self.correct_way = nn.Parameter(torch.randn(3, 3))      # Registered
    def forward(self):
        print("Tensor device: ",self.wrong_way.device)
        print("Parameter device: ",self.correct_way.device)

model = MyModel()
model.to('cuda')
model()
print("Trainable parameters:")
for p in model.parameters():
    print(p.shape)

Tensor device:  cpu
Parameter device:  cuda:0
Trainable parameters:
torch.Size([3, 3])


**Drawbacks**
 - It only capture one meaning of the sentence, but a sentence can have multiple meaning.

## Multihead Causal Attention
- The main aim of Multihead Causal attention is to capture multiple meanings of a sentence.
- The meaning which is more accurate with respect to the context get more attention weights.
- There 2 ways of implementing Multihead Causal attention: 
    1. Combining multiple Causal Attention
    2. Split attention weights(Recommended)
- We use the parameter `num_heads` to capture these many number of meanings.

### Combining Multiple Causal Attention
**Algorithm**
1. Create a list of `CausalAttention` layers in the `__init__` function
2. Concatenate the output of each layer

In [17]:
class MultiheadAttentionV1(nn.Module):
    def __init__(self,dim,dropout,context_length,num_heads,qkv_bias=False):
        super().__init__()
        self.heads=nn.ModuleList(
            [CausalAttention(dim,dropout,context_length,qkv_bias)
             for _ in range(num_heads)]
        )
    def forward(self,x):
        return torch.cat([head(x) for head in self.heads],dim=-1)

In [18]:
mha1=MultiheadAttentionV1(input_batch.shape[2],0.1,input_batch.shape[1],3)
logits1=mha1(input_batch)
print('Output dimention: ',logits1.shape[2])

Output dimention:  9


- As you can see in the code above, the output dimention of the context vector is $\text{dim}\times\text{num heads}=9$, and as we pass the input through multiple number of similar attentional layer, as we do in GPT , the embedding dimention increases exponentially consuming unneccessery space.

**Drawbacks**
- Exponential increase in dimention size as we pass through multiple layers

### Spliting Attention Weights(Recommended)
- This is a better way of implementation because there is no change in output dimention.
- Here we divide the dimention to attention heads.So the dimention(`dim`) must be divisable with `num_heads`.
- We pass the context vectors to the trainable output project layer. The main purpose of project layer is to assign weights or importance to different meanings of a word captured by different heads, and combine them to give the exact meaning of a word with respect the sentence given the context.

**Algorithm**
- In `__init__()` function:
    1. Check if `dim` is divisable by `num_heads`
    2. Declare Dimention,Number of heads and head dimention
    3. Declare `W_q`,`W_k` and `W_v`.
    4. Declare output projection,dropout and mask
- In `forward()` function:
    1. Find dimentions of input
    2. Find queries,keys and values by passing through `W_q`,`W_k` and `W_v`.
    3. Splitting dimention over heads: dim -> [num_heads,head_dim]
    4. Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim) to find attention scores of each head
    5. Find attention scores of each head and apply mask
    6. Find attention weights of each head
    7. Find context vector of each head
    8. Combine context vector of each head and apply projection layer

In [ ]:
class MultiheadAttentionV2(nn.Module):
    def __init__(self,dim,dropout,num_heads,context_length,qkv_bias=False):
        super().__init__()
        # Check if `dim` is divisable by `num_heads`
        assert dim % num_heads==0,\
        "dim must be divisible by num_heads"
        # Declare Dimention,Number of heads and head dimention
        self.dim=dim
        self.num_heads=num_heads
        self.head_dim=dim//num_heads
        # Declare `W_q`,`W_k` and `W_v`.
        self.W_q=nn.Linear(dim,dim,bias=qkv_bias)
        self.W_k=nn.Linear(dim,dim,bias=qkv_bias)
        self.W_v=nn.Linear(dim,dim,bias=qkv_bias)
        # Declare output projection,dropout and mask
        self.out_proj=nn.Linear(dim,dim)
        self.dropout=nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length,context_length),
                       diagonal=1)
        )
    def forward(self,x):
        # Find dimentions of input
        b,num_tokens,dim=x.shape
        # Find queries,keys and values by passing through `W_q`,`W_k` and `W_v`.
        queries=self.W_q(x)
        keys=self.W_k(x)
        values=self.W_v(x)
        # Splitting dimention over heads: dim -> [num_heads,head_dim] 
        queries=queries.view(b,num_tokens,self.num_heads,self.head_dim)
        keys=keys.view(b,num_tokens,self.num_heads,self.head_dim)
        values=values.view(b,num_tokens,self.num_heads,self.head_dim)
        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim) 
        queries=queries.transpose(1,2)
        keys=keys.transpose(1,2)
        values=values.transpose(1,2)
        # Find attention scores of each head and apply mask
        attn_scores=queries@keys.transpose(2,3)
        attn_scores=attn_scores.masked_fill(self.mask.bool(),-torch.inf)
        # Find attention weights of each head
        attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
        attn_weights=self.dropout(attn_weights)
        # Find context vector of each head
        context_vectors=(attn_weights@values).transpose(1,2)
        # Combine context vector of each head and apply projection layer
        context_vectors=context_vectors.contiguous().view(b,num_tokens,self.dim)
        context_vectors=self.out_proj(context_vectors)
        return context_vectors
torch.manual_seed(123)
mha2=MultiheadAttentionV2(input_batch.shape[2],0.1,3,input_batch.shape[1])
logits2=mha2(input_batch)  
print(logits2)

tensor([[[-0.7344, -0.2141, -0.4846],
         [-1.2381, -0.7748, -0.9687],
         [-0.9554, -0.6227, -0.7664],
         [-0.9483, -0.7925, -0.8409],
         [-0.9393, -0.4574, -0.6384],
         [-0.8615, -0.2996, -0.6141]],

        [[-0.7344, -0.2141, -0.4846],
         [-1.2381, -0.7748, -0.9687],
         [-0.9030, -0.5601, -0.7362],
         [-0.9798, -0.7094, -0.9149],
         [-0.9128, -0.4700, -0.6346],
         [-0.9233, -0.4810, -0.6735]]], grad_fn=<ViewBackward0>)
